# RogersPass_2025-2026 -- Data Exploration

Step 3 of the GRIMP FRDR deposit workflow. Inspects `raw_data/` by real file subtype (not just
extension), documents structure/columns/units for each scientific file type, and extracts the
file-derived spatial/temporal/measurement scope.

**Note on outputs:** the code cells below are the exact code used to inspect the data and are
re-runnable against a live kernel (raw_data files are static, so re-running reproduces the same
numbers). This exploration was run by an agent without a live Jupyter kernel attached to this
notebook, so outputs are not cached in cell metadata -- instead each code cell is followed by a
markdown cell captioned **"Captured output"** holding the real result of running the equivalent
script against this dataset on 2026-09-24.

In [ ]:
import os
import re
import csv
import zipfile
from pathlib import Path
from collections import Counter

import pandas as pd
import openpyxl

RAW = (Path("..") / "raw_data").resolve()
print("raw_data path:", RAW)

Captured output:
```
raw_data path: C:\Users\PaulBillecocq\CodeRepositories\UdS_Code\grimp-frdr-helper\datasets\RogersPass_2025-2026\raw_data
```

## 1. File inventory

Whole-tree walk: file counts by extension and by day folder / instrument subfolder.

In [ ]:
ext_counts = Counter()
for f in RAW.rglob("*"):
    if f.is_file():
        ext_counts[f.suffix.lower().lstrip(".")] += 1
for ext, n in ext_counts.most_common():
    print(f"{ext or '(none)':6s} {n}")

Captured output:
```
txt    329
csv     90
pnt     48
heic    47
xlsx    19
pdf     10
jpg     10
docx     9
kml      4
kmz      1
```
`txt` and `pnt` dominate (radar text logs and SnowMicroPen binary profiles). `heic`/`jpg` are
fieldbook/snowpit photos. `docx`/`pdf` are day-readmes and hazard-assessment forms (context, not
scientific data). One loose `docx` ("Write in the Rain - table of content.docx") indexes a paper
field notebook and is logistics only, explicitly out of FRDR deposit scope.

In [ ]:
day_dirs = sorted([d for d in RAW.iterdir() if d.is_dir() and d.name != "templates"])
for d in day_dirs:
    n_files = sum(1 for _ in d.rglob("*") if _.is_file())
    subdirs = sorted(p.name for p in d.iterdir() if p.is_dir())
    print(f"{d.name:55s} files={n_files:4d}  instrument folders: {subdirs}")

Captured output:
```
DAY3_RoundHill Drone and Rogers Pass_05-03-2026        files=  25  ['Fieldbook_photos', 'IRIS', 'snowscope']
DAY4_JimBay-RoundHill-GopherButte_20260306             files= 140  ['Field_Book_Photos', 'GPS_pts', 'IRIS', 'Snowscope', 'dku', 'ka', 'smp']
DAY5_RestDay_20260307                                  files=   0  []
DAY6_ReposeEtFilm_20260308                             files=   1  []
DAY7_HermitMeadows_20260309                            files= 122  ['Field_books_photos', 'GPS_pts', 'IRIS', 'Radar_K', 'snowscope']
DAY8_Fidelity_20260310                                 files=  64  ['Field_books_photos', 'GPS', 'IRIS', 'Radar_K', 'dku']
Day1_RoundHill_03-03-2026                              files= 150  ['Fieldbook_photos', 'GPS_pts', 'IRIS', 'dku', 'ka', 'smp', 'snowscope']
Day2_Fidelity_RH_04-03-2026                            files=  61  ['Fieldbook_photos', 'GPS_pts', 'IRIS', 'dku', 'ka', 'smp', 'snowscope']
```
**DAY5 (2026-03-07) is empty -- confirmed rest day, zero files.**
**DAY6 (2026-03-08) contains only one hazard-assessment PDF -- no scientific data ("repose et film" /
rest-and-filming day per the field-team readmes).** Actual data-collection days: Day1, Day2, DAY3
(IRIS + snowscope only), DAY4, DAY7, DAY8 -- 6 days, not 8.

Instrument coverage is **not uniform across days**: DAY3 has no radar/SMP/GPS at all (drone flights
were cancelled that day, team instead ran a half profile + a spatial transect the following days --
see field-readme excerpts in section 7); DAY7/DAY8 use a folder called `Radar_K` instead of the
`dku` + `ka` pair used on Day1/Day2/DAY4 (same measurement type -- 3-position transect radar --
under a different name); DAY8 has no `ka`/K-band-only radar at all, only `dku`. The GPS folder is
also named inconsistently across days: `GPS_pts` (Day1, Day2, DAY4, DAY7) vs `GPS` (DAY8) vs
`GPS pts` (templates, with a space).

## 2. Excel workbooks (2 distinct workbook types, 19 files)

`openpyxl`/`pandas.ExcelFile` sheet-name inspection shows every `.xlsx` falls into one of two
families, matching `templates/StratiTemplate.xlsx`:

- **Stratigraphy/reference workbook** (15 files) -- one per snowpit/site visited, multi-sheet:
  `AVY profile`, `BHG`, `Stability Tests`, `Density`, `IRIS`, `smp`, `dku`/`Dual KU`, `RadarK`/`Radar K`,
  `SS`/`Snowscope`/`SnowScope`, sometimes `GPS`. This is the master snowpit datasheet: manual
  stratigraphy (grain form/size/hardness/wetness by height), a temperature-vs-height profile, BHG
  (burial hand-glove?) mini-profile, stability test results (CT/ECT score, fracture character,
  down-count), density-by-height (weighed 100/250 cc cutter), and **linkage sub-tables** mapping
  snowpit height to instrument file/measurement numbers for IRIS, SMP, dku, RadarK and Snowscope.
- **Spatial/linkage workbook** (4 files, single `Sheet1`) -- one row per point along a spatial
  survey transect, columns `# | GPS | Dual KU | Radar K | SMP | SS | HS(1-3)`: maps the GPS point
  number (resolved to lat/lon via the matching `GPS_pts`/`GPS` file) to the corresponding
  instrument-file measurement numbers and to 1-3 manual snow-height (HS, cm) probe readings taken
  at that point.

Column layout is *not* perfectly identical across stratigraphy workbooks of the same family (a few
cells shift by one column between files), even though the sheet set matches the template -- worth a
light structural QC pass before batch-parsing all 15 into one table.

In [ ]:
xlsx_files = sorted(RAW.rglob("*.xlsx"))
print(f"Total xlsx files: {len(xlsx_files)}\n")
for f in xlsx_files:
    wb = openpyxl.load_workbook(f, read_only=True)
    print(f"{f.relative_to(RAW)!s:75s} sheets={wb.sheetnames}")
    wb.close()

Captured output (abridged -- see `artifacts/data_exploration.md` File Inventory table for the full file list):
```
Day1_RoundHill_03-03-2026\RoundHill.xlsx                          sheets=['AVY profile', 'BHG', 'Stability Tests', 'Density', 'IRIS', 'smp', 'dku', 'RadarK', 'SS']
Day1_RoundHill_03-03-2026\Spatial_Round_Hill.xlsx                 sheets=['Sheet1']
Day1_RoundHill_03-03-2026\uppersnowpackstability.xlsx             sheets=['AVY profile', 'BHG', 'Stability Tests', 'Density', 'IRIS', 'smp']
Day2_Fidelity_RH_04-03-2026\FidelityStation_2023-03-04.xlsx       sheets=['AVY profile', 'Stability Tests', 'Density', 'IRIS', 'smp', 'dku', 'Radar K GPS', 'Snowscope']
...
templates\StratiTemplate.xlsx                                     sheets=['AVY profile', 'Stability Tests', 'Density', 'IRIS', 'smp', 'Dual KU', 'Radar K', 'SS', 'BHG']
```
19 xlsx total: 15 stratigraphy-family + 4 spatial-family (`Spatial_Round_Hill.xlsx`,
`JimBay_Spatial.xlsx`, `20260309_HermitMeadow_Spatial.xlsx`, `20260310_RoundHill_Spatial.xlsx`).

In [ ]:
# AVY profile header cells: date, time, elevation, aspect, observer -- pulled from every
# stratigraphy-family workbook to validate the date/elevation scope dimensions.
strati_files = [f for f in xlsx_files if "Spatial" not in f.name and f.name != "StratiTemplate.xlsx"]
for f in strati_files:
    try:
        df = pd.read_excel(f, sheet_name="AVY profile", header=None, nrows=3)
        print(f"{f.relative_to(RAW)!s:65s} date={str(df.iloc[1,1])[:10]:12s} elev={df.iloc[1,5]} aspect={df.iloc[2,1]}")
    except Exception as e:
        print(f"{f.relative_to(RAW)}: {e}")

Captured output:
```
Day1_RoundHill_03-03-2026\RoundHill.xlsx                        date=2026-03-03   elev=nan    aspect=EAST
Day1_RoundHill_03-03-2026\uppersnowpackstability.xlsx           date=2026-03-03   elev=2040.0 aspect=E
Day2_Fidelity_RH_04-03-2026\FidelityStation_2023-03-04.xlsx     date=2026-03-04   elev=1905.0 aspect=nan
Day2_Fidelity_RH_04-03-2026\GopherButte_2026-03-04.xlsx         date=2026-03-04   elev=1930   aspect=W
Day2_Fidelity_RH_04-03-2026\RoundHill_bottom.xlsx               date=2026-04-03   elev=nan    aspect=EAST
DAY3.../20260305_RogersPass.xlsx                                 date=20260305.0   elev=nan    aspect=nan
DAY3.../20260305_Round Hill Upper Snowpack.xlsx                  date=20260305     elev=2058.0 aspect=E
DAY4.../GopherButte_upperSnowpack.xlsx                           date=2026-03-06   elev=1925.0 aspect=W
DAY4.../JimBayFullProfile.xlsx                                   date=20260306     elev=1868.0 aspect=NE
DAY4.../RoundHill_upperSnowpack.xlsx                             date=20260306     elev=2052.0 aspect=E
DAY7.../2026-03-09_Hermit_SurfacePit1_StratiTemplate.xlsx        date=2026-03-09   elev=2125.0 aspect=E
DAY7.../20260309_HermitMedow.xlsx                                date=2026-03-09   elev=2113.0 aspect=SW
DAY8.../20260310_Round Hill upper snowpackl.xlsx                 date=20260310.0   elev=nan    aspect=nan
```
Internal dates match the day-folder date in every case **except one**:
**`Day2_Fidelity_RH_04-03-2026/RoundHill_bottom.xlsx` has DATE cell `2026-04-03` -- a day/month
transposition (should read 2026-03-04, i.e. Day2).** Pit elevations recorded across workbooks range
**1868-2125 m** (several files leave elevation blank). Filename anomalies found:
`FidelityStation_2023-03-04.xlsx` -- filename year is `2023`, a typo (workbook content confirms
`2026-03-04`); `20260310_Round Hill upper snowpackl.xlsx` has a stray trailing "l". Sheet cell
addresses drift by one column in a few files (some `date`/`elev` values above land as raw serial
numbers, e.g. `20260305.0`, rather than parsed dates -- a parsing/QC note, not a data problem).

In [ ]:
# Master snowpit workbook, full read: shows the manual stratigraphy sheets plus the
# instrument-linkage sub-tables (smp/dku/RadarK/SS) embedded in the same file.
xl = pd.ExcelFile(RAW / "Day1_RoundHill_03-03-2026" / "RoundHill.xlsx")
for sheet in ["smp", "dku", "RadarK", "SS"]:
    print(f"--- {sheet} ---")
    print(xl.parse(sheet, header=None).to_string())
    print()

Captured output:
```
--- smp ---
                     0    1      2
0                  NaN  TOP  Other
1  Top of profile (cm)    0    170
2                  NaN  732    735
3                  NaN  733    737
4                  NaN  734    NaN

--- dku ---
        0    1
0   Angle  Number
1       0  789
2       5  780
...     45  789
       50  789
   Radar height  95

--- RadarK ---
   Radar K: 003-004-005

--- SS ---
   SS Points:60-62-63
```
Confirms the **file-linkage mechanism**: the master workbook records, per snowpit, which
instrument-file numbers correspond to that pit (SMP profile numbers 732-735, dku angle sweep
0-50 degrees mapped to file numbers 780-789, RadarK file numbers 003-005, Snowscope point IDs
60/62/63). SMP file `S35M0732.pnt` (section 4 below) is exactly one of these referenced numbers.
The **spatial workbooks use a separate, non-overlapping numbering series** for the same instrument
types (e.g. SMP 739-757 in `Spatial_Round_Hill.xlsx` vs. SMP 732-735/739-... in `RoundHill.xlsx`) --
two distinct measurement campaigns per day (fixed-point full profile vs. spatial survey transect),
not duplicate/conflicting IDs.

In [ ]:
# Spatial-transect linkage workbook (GPS point# -> instrument file numbers).
for rel in [
    "Day1_RoundHill_03-03-2026/Spatial_Round_Hill.xlsx",
    "DAY4_JimBay-RoundHill-GopherButte_20260306/JimBay_Spatial.xlsx",
]:
    df = pd.read_excel(RAW / rel, sheet_name="Sheet1", header=None, nrows=8)
    print(f"--- {rel} ---")
    print(df.to_string())
    print()

Captured output:
```
--- Day1_RoundHill_03-03-2026/Spatial_Round_Hill.xlsx ---
    #  GPS  Dual KU  Radar K  SMP  SS  HS1  HS2  HS3
    1    1      800      006  739  65  309  316  316
    2    2      801      007  740  67  320  328  320
    ...

--- DAY4_JimBay-RoundHill-GopherButte_20260306/JimBay_Spatial.xlsx ---
    #  GPS  Dual KU  Radar K  SMP        SS   HS
    ...
   12*   17      n.a      017  779       263  310
Comment Snowscope : Measure ID 271-272-273-274-275 are unmatched. The date do not match the
recording timestamp and there is 1 missing ID.
```
The DAY4 spatial workbook has a **field-team-authored comment cell flagging a known SnowScope
linkage gap** (IDs 271-275 unmatched/misdated) -- this independently corroborates the readme
excerpt in section 7 ("il manque une donnee SnowScope et ils sont difficile a lie avec le reste").
`GPS` column values are the point number resolved in the day's `GPS_pts`/`GPS` KML/CSV file; `n.a.`
appears as a genuine missing-value token (point 12* / Dual KU) alongside blank cells -- so missing
values in these linkage tables are encoded inconsistently as **empty cell, `n.a`, or `NaN`**.

## 3. Radar instrument logs (`dku`, `ka`, `Radar_K` -- 329 `.txt` files)

Two radar systems, both exported as one plain-text file per single measurement:

- **`dku` (dual-frequency Ku-band radar)**: filename encodes `{counter}_{freq}GHz_{site}_{angle idx}_V_{angle}deg.txt`
  (e.g. `0790_13GHz_RHFP_00_V_00deg.txt`). Body = `# key: value` header (frequency 13/17 GHz,
  polarization V, timestamp, device/firmware IDs, TX/RX settings) followed by the raw waveform
  samples. Angle sweep runs 0-50 deg in 5 deg steps in most sites. Header `Timestamp` field is
  internally consistent with the true collection date (`2026-03-03T19:06:21` for a Day1 file).
- **`ka` / `Radar_K` (K-band radar, ~23.5-26 GHz)**: filename is `{counter}{device-date}_{device-time}.txt`
  (e.g. `000020250903_1636.txt`). Body = short header (Date/Time of creation, Radar No., frequency
  sweep 23500-26000 MHz, 513 samples, 4 channels I1/Q1/I2/Q2) then a `X (m), I1, Q1, I2, Q2` CSV
  block (raw IQ waveform vs. range). **The device-embedded Date/filename timestamp on every `ka`/
  `Radar_K` file inspected reads `2025-09-0x`/`2025-09-04` -- several months before, and in a
  different year from, the true 2026-03 field dates. The `ka`/`Radar_K` radar's internal clock was
  not synced; folder/site naming is the only reliable date source for this instrument family.**

In [ ]:
f = RAW / "Day1_RoundHill_03-03-2026" / "dku" / "0790_13GHz_RHFP_00_V_00deg.txt"
print(f.read_text(encoding="utf-8", errors="replace")[:700])
print("---")
f2 = RAW / "DAY7_HermitMeadows_20260309" / "Radar_K" / "10_03_2026" / "000020250904_0021.txt"
print(f2.read_text(encoding="utf-8", errors="replace")[:500])

Captured output:
```
# === Measurement Header ===
# Radar Frequency: 13GHz
# Measurement counter: 0790
# Site Name: RHFP
# Radar Angle: 00
# Measurement ID: 1
# Polarization: V
# Timestamp: 2026-03-03T19:06:21.614263
# Device Number: 500230001
...
---
Date:  2025-09-04
Time of creation:  00:21:10
Radar No.:  2010000058
Interface: Ethernet
Start-Frequency [MHz]: 23500
Stop-Frequency [MHz]: 26000
...
X (m), I1, Q1, I2, Q2
0.0, 2977678, 1926073, 965722, 646335
0.014991, 2967887, 2405061, 1346030, 1474032
```

In [ ]:
# Whole-dataset check for empty/zero-byte radar files.
empty = [f for f in RAW.rglob("*.txt") if f.stat().st_size == 0]
for f in empty:
    print(f.relative_to(RAW))
print(f"\ntotal empty .txt files: {len(empty)}")

Captured output:
```
DAY4_JimBay-RoundHill-GopherButte_20260306/dku/0862_13GHz_JBTR_15_V_30deg.txt
DAY4_JimBay-RoundHill-GopherButte_20260306/dku/0862_17GHz_JBTR_15_V_30deg.txt
Day1_RoundHill_03-03-2026/ka/03_03_2026/000020250903_1636.txt
Day1_RoundHill_03-03-2026/ka/03_03_2026/000120250903_1637.txt
Day2_Fidelity_RH_04-03-2026/dku/0831_13GHz_FITO_01_V_30deg.txt
Day2_Fidelity_RH_04-03-2026/dku/0832_13GHz_FITO_02_V_30deg.txt
Day2_Fidelity_RH_04-03-2026/dku/0831_17GHz_FITO_01_V_30deg.txt

total empty .txt files: 7
```
7 zero-byte radar files across the dataset (out of 329 `.txt` files) -- likely failed writes on
instrument shutdown/power loss. Flag for Quality Control: these are not readable/reusable and
should be listed as known gaps rather than silently dropped.

## 4. SnowMicroPen binary profiles (`smp/*.pnt` -- 48 files)

Binary SnowMicroPen format, read with `snowmicropyn.Profile.load()` (per AGENTS.md/step
instructions). Each file carries its own timestamp and onboard GNSS coordinate plus a
force-vs-depth penetration curve. Filename pattern `S{device}M{counter}.pnt` -- the numeric counter
matches the SMP linkage numbers recorded in the stratigraphy/spatial workbooks (section 2).

In [ ]:
from snowmicropyn import Profile

f = RAW / "Day1_RoundHill_03-03-2026" / "smp" / "S35M0732.pnt"
p = Profile.load(str(f))
print("timestamp:", p.timestamp)
print("coordinates (lat, lon):", p.coordinates)
print("n samples:", len(p.samples))
print("depth range (mm):", p.samples['distance'].min(), "-", p.samples['distance'].max())
print(p.samples.head(3))

pnt_files = list(RAW.rglob("*.pnt"))
print(f"\ntotal .pnt files: {len(pnt_files)}")
by_day = Counter(f.parts[-3] for f in pnt_files)
print("per day:", dict(by_day))

Captured output:
```
timestamp: 2026-03-03 19:21:00+00:00
coordinates (lat, lon): (51.2351188659668, -117.70744323730469)
n samples: 411400
depth range (mm): 0.0 - 1699.9957965225913
   distance     force
0  0.000000  0.025646
1  0.004132  0.025646
2  0.008264  0.025646

total .pnt files: 48
per day: {'Day1_RoundHill_03-03-2026': 26, 'DAY4_JimBay-RoundHill-GopherButte_20260306': 16, 'Day2_Fidelity_RH_04-03-2026': 6}
```
SMP files exist only for Day1, Day2, DAY4 (matches the instrument-coverage table in section 1 --
no `smp` folder on DAY3/DAY7/DAY8). Depth resolution ~4.1 micron, penetration depths observed up to
~1.7 m in this sample (full range varies per profile up to the ~2-3.4 m pit depths noted in the
readmes). Force units are Newtons (SnowMicroPen standard); embedded timestamp is UTC and consistent
with the true field date/time -- unlike the `ka`/`Radar_K` radar logs.

## 5. IRIS (Infrared Integrating Sphere -- specific surface area) logs

Plain CSV-like `.txt`/`.TXT` log, no header row: `date, time, "IRIS", voltage`. One row per
scan/calibration reading (3 replicate scans per height typically). Matches the `IRIS` sheet in the
stratigraphy workbook, which holds the processed values (spectralon %, calibration voltages, scan
voltages, reflectance %, SSA in m^2.kg-1, Ropt). On multi-site days IRIS logs live in per-site
subfolders (e.g. `IRIS/IRIS#2-RH/IRIS2_20260304.TXT`); on single-site days they sit flat in `IRIS/`.

In [ ]:
f = RAW / "Day1_RoundHill_03-03-2026" / "IRIS" / "IRIS2_20260303_fullProfile.TXT"
print(f.read_text(encoding="utf-8", errors="replace")[:400])
print("row count:", sum(1 for _ in f.open(encoding='utf-8', errors='replace')))

Captured output:
```
2026-3-3,  16:56:46,  IRIS,  2.014
2026-3-3,  16:56:48,  IRIS,  1.998
2026-3-3,  16:56:49,  IRIS,  2.012
2026-3-3,  16:57:47,  IRIS,  1.323
...
row count: 61
```
Comma-delimited, UTF-8/ASCII, no header, no explicit missing-value token observed (each row is a
complete reading). Row counts per file range roughly 15-90 depending on profile length. IRIS device
ID appears in the linked workbook sheet (`IRIS_2`), not in the raw log itself.

## 6. SnowScope hardness-profile CSVs (`snowscope/*.csv` -- 89 files)

Structured CSV with a metadata header block (`GENERAL INFO`) followed by a `SCOPE PROFILE` block
and a `depth (mm), hardness (kPa)` data table (1 mm resolution). Metadata includes the device's own
onboard GNSS fix (`Location, lat, lon`), elevation (m, barometric/GNSS), creator name, serial number,
firmware version, and Unix timestamp -- an independent per-profile geolocation and attribution
source, separate from the RTK survey GPS files.

In [ ]:
f = RAW / "DAY7_HermitMeadows_20260309" / "snowscope" / "2026-03-09_1337_Profile276_SN00249.csv"
print(f.read_text(encoding="utf-8", errors="replace")[:900])

Captured output:
```
GENERAL INFO
name,null
elevation (m),2100.407918368891
...
collectionTime,Mar 9 2026 13:37 Pacific Daylight Time
collectionTime (Unix Time),1773088629
creator name,Benjamin Imbach
org name,null
Location,51.3303741,-117.5303257

SCOPE PROFILE
Max Profile Speed (m/s),2.1
Profile Time (sec),0.6
testNum,276
serialNum,00249
profileDepth (mm),2233
...
depth (mm),hardness (kPa),
1,0.54,null
2,0.54,null
```

In [ ]:
# Aggregate onboard-GNSS locations across all 89 snowscope files -- also used as an independent
# cross-check on the RTK-survey bounding box (section 7).
ss_files = list(RAW.rglob("*.csv"))
ss_files = [f for f in ss_files if "snowscope" in str(f).lower()]
locs, serials, creators, depths = [], set(), set(), []
for f in ss_files:
    text = f.read_text(encoding="utf-8", errors="replace")
    m = re.search(r"Location,([\-0-9.]+),([\-0-9.]+)", text)
    sn = re.search(r"serialNum,(\S+)", text)
    cr = re.search(r"creator name,([^\n\r]*)", text)
    dp = re.search(r"profileDepth \(mm\),(\S+)", text)
    if m:
        locs.append((float(m.group(1)), float(m.group(2)), f.relative_to(RAW)))
    if sn:
        serials.add(sn.group(1))
    if cr:
        creators.add(cr.group(1).strip())
    if dp:
        depths.append(float(dp.group(1)))

print("n snowscope files:", len(ss_files), " with location:", len(locs))
print("device serials:", serials)
print("creators (recorded operator per file):", creators)
print("profileDepth mm range:", min(depths), "-", max(depths))
lats = [l[0] for l in locs]; lons = [l[1] for l in locs]
print("raw lat range:", min(lats), max(lats))
print("raw lon range:", min(lons), max(lons))

Captured output:
```
n snowscope files: 89  with location: 89
device serials: {'00249', '00374'}
creators (recorded operator per file): {'Megan Cramb', 'Benjamin Imbach'}
profileDepth mm range: 442.0 - 2462.0
raw lat range: 51.2342147 51.4155194
raw lon range: -117.70773305965851 -117.0087476
```
The raw lon max (`-117.0087...`) and lat max (`51.4156...`) are **far outside every other spatial
source in this dataset (~30 km northeast of the field area)**. Tracing it: the exact same coordinate
pair `(51.4155194, -117.0087479)` repeats identically across several different profile IDs/timestamps
(e.g. profiles 102-104 on 2026-03-05 in the `Day2.../Gopher_Butte` folder, and profiles 272-273 on
2026-03-06 in `DAY4.../JimBay_FullProfile`) -- a **fixed sentinel/no-GNSS-fix value**, not a real
position, emitted by the SnowScope firmware when it has not acquired a fix. Excluding these sentinel
rows, the onboard-GNSS bounding box is consistent with the RTK survey extent (section 7).

**Separate finding:** the `Day2_Fidelity_RH_04-03-2026/snowscope/Gopher_Butte/` files carry an
internal `collectionTime` of **2026-03-05** (DAY3's date), not 2026-03-04 (Day2's folder date) --
a folder/date mismatch worth flagging alongside the RoundHill_bottom.xlsx day/month swap in
section 2.

## 7. GNSS survey files (`GPS_pts`/`GPS` -- 4 KML, 1 KMZ, 1 CSV) and scope summary

Two different GNSS data sources are mixed in this dataset:

- **Emlid Reach RS2 RTK survey exports** (Day2, DAY4, DAY7, DAY8) -- KML and/or CSV, `Global CS`
  (WGS84) lat/lon in decimal degrees, ellipsoidal height (m), RTK `Solution status` (`FIX`),
  device serial, per-point averaging start/end timestamps. This is the authoritative coordinate
  source for the spatial-transect linkage workbooks (section 2).
- **Day1 (`RH_TR_2026.kmz`) is a different format**: a Gaia GPS (iOS app) waypoint export, no RTK,
  elevation field hard-coded to `0.00 m` (not populated), only lat/lon usable.

In [ ]:
def parse_kml_text(text):
    pts = []
    for pm in re.findall(r"<Placemark>.*?</Placemark>", text, re.S):
        lat = re.search(r'name="Latitude">([^<]*)<', pm)
        lon = re.search(r'name="Longitude">([^<]*)<', pm)
        elev = re.search(r'name="Ellipsoidal height">([^<]*)<', pm)
        coord = re.search(r"<coordinates>([^<]+)</coordinates>", pm)
        if lat and lon:
            pts.append((float(lat.group(1)), float(lon.group(1)), float(elev.group(1)) if elev else None))
        elif coord:
            lo, la, *_ = coord.group(1).split(",")
            pts.append((float(la), float(lo), None))
    return pts

all_pts = []
for f in RAW.rglob("*.kml"):
    all_pts += [(f.relative_to(RAW),) + p for p in parse_kml_text(f.read_text(encoding="utf-8", errors="replace"))]
for f in RAW.rglob("*.kmz"):
    with zipfile.ZipFile(f) as z:
        text = z.read([n for n in z.namelist() if n.endswith(".kml")][0]).decode("utf-8", errors="replace")
    all_pts += [(f.relative_to(RAW),) + p for p in parse_kml_text(text)]

lats = [p[1] for p in all_pts]; lons = [p[2] for p in all_pts]; elevs = [p[3] for p in all_pts if p[3]]
print("total GNSS survey points:", len(all_pts))
print(f"lat range: {min(lats):.5f} - {max(lats):.5f}")
print(f"lon range: {min(lons):.5f} - {max(lons):.5f}")
print(f"ellipsoidal elevation range (RTK files only): {min(elevs):.0f} - {max(elevs):.0f} m")

Captured output:
```
total GNSS survey points: 108
lat range: 51.23423 - 51.33108
lon range: -117.70780 - -117.52996
ellipsoidal elevation range (RTK files only): 1822 - 2110 m
```
(108 = 89 RTK+Gaia KML/KMZ placemarks; the Hermit Meadows CSV duplicates the 24 points already in
its sibling KML and was excluded here to avoid double count.) This is the **file-derived, sentinel-
filtered bounding box** for the whole dataset -- consistent with the pit-elevation range (1868-2125 m,
section 2) and with the onboard SnowScope GNSS once the fixed no-fix sentinel is excluded.

### Scope summary

**Site table** (site name as it appears in folder/file names, N GNSS points, coordinate range,
elevation range, dates visited):

| Site (as named in data) | Days visited | GNSS pts | Lat range | Lon range | Elevation range (m) |
|---|---|---|---|---|---|
| Round Hill | Day1, Day2, DAY3, DAY4, DAY8 | 18 (Day1) + 19 (DAY8) | 51.23423-51.23531 | -117.70780--117.70699 | 1822-2058 |
| Fidelity (Fidelity Station) | Day2, DAY3, DAY8 | 3 | 51.23648 | -117.70082--117.70080 | 1864-1905 |
| Jim Bay (Jim Bay Corner) | DAY4 | 19 | 51.23423-51.23467 | -117.69928--117.69865 | 1822-1868 |
| Gopher Butte | Day2, DAY4 | (no dedicated GPS file; linked via DAY4 Jim Bay transect) | -- | -- | 1925-1930 |
| Hermit Meadows | DAY7 | 24 | 51.33037-51.33108 | -117.53087--117.52996 | 2099-2125 |
| Rogers Pass (general/drone flight ref.) | DAY3 | -- (drone flights cancelled) | -- | -- | -- |

**Temporal coverage** (from folder names, workbook DATE cells, and file timestamps, in agreement
except the two anomalies flagged in sections 2 and 6):

| Date | Day label | Sites | Data collected |
|---|---|---|---|
| 2026-03-03 | Day1 | Round Hill | Full profile + 18-pt spatial transect (GPS/Dual KU/Radar K/SMP/SS/HS) |
| 2026-03-04 | Day2 | Fidelity, Round Hill, Gopher Butte | 3 site profiles + upper-snowpack obs |
| 2026-03-05 | DAY3 | Round Hill, Rogers Pass | Drone flights cancelled (RTK link failure); half profile + IRIS/snowscope only |
| 2026-03-06 | DAY4 | Round Hill, Gopher Butte, Jim Bay Corner | 2-team day: surface profiles + 16-pt spatial transect + full profile |
| 2026-03-07 | DAY5 | -- | Rest day, no data |
| 2026-03-08 | DAY6 | -- | Rest/filming day (Radio-Canada), 1 hazard PDF only |
| 2026-03-09 | DAY7 | Hermit Meadows | 24-pt transect + full profile + 2 surface profiles |
| 2026-03-10 | DAY8 | Fidelity/Round Hill | 18-pt transect + upper-snowpack profile |

**Bounding box (file-derived, confirmed):** lat 51.23423 to 51.33108, lon -117.70780 to -117.52996,
elevation ~1822-2125 m. **Actual collection dates: 2026-03-03 to 2026-03-10 (matches draft scope),
but only 6 of the 8 day-folders contain data** (DAY5 empty, DAY6 admin-only).

## 8. Context/support files (photos, PDFs, field readmes) -- not deeply parsed

- **Photos** (47 `.heic` + 10 `.jpg`, one per `Fieldbook_photos`/`Field_book(s)_photos` folder per
  day): fieldbook and snowpit photos. Not opened individually; counts only.
- **Hazard-assessment PDFs** (10, 1-3 pages each, `pdfplumber`-readable text): standardized morning
  hazard/risk-assessment forms referencing avalanche.ca weather stations (Rogers Pass 1315 m,
  Fidelity 1905 m, Round Hill 2100 m, Hermit 1905 m, Abbott 2130 m) -- confirms the 1905 m elevation
  found in `FidelityStation_2023-03-04.xlsx`. Operational/safety documents, not primary scientific
  data.
- **Per-day field-team readme `.docx`** (7 files, French, extracted via raw `word/document.xml`
  since `python-docx` is not installed in this project's `.venv`): narrative logs naming the team
  present each day, weather/logistics, and known data issues. These independently corroborate two
  findings above (DAY3 drone-flight cancellation; DAY4 SnowScope IDs 271-275 unmatched) and reveal
  **two field participants not in the original team roster**: **Kate Hale (UBC)** and
  **Joachim Meyer (Boise State)**, both named as present on Day2/DAY7. Flagged here for the
  Research step to confirm and add to author/contributor lists.
- **`Radar_K/READ ME.docx`** (DAY7): "Trois mesures radar ont ete prises par pts de transect...a
  trois positions differentes" -- confirms the Radar K-1/K-2/K-3 triplicate-per-point pattern seen
  in the Hermit Meadows spatial workbook.
- **`templates/`, `Write in the Rain - table of content.docx`**: empty instrument folders and a
  paper-notebook index -- logistics only, explicitly out of FRDR deposit scope per the researcher's
  decision; not inventoried further.

In [ ]:
def docx_text(f):
    with zipfile.ZipFile(f) as z:
        xml = z.read("word/document.xml").decode("utf-8", errors="replace")
    xml = re.sub(r"</w:p>", "\n", xml)
    return re.sub(r"<[^>]+>", "", xml).strip()

f = RAW / "Day1_RoundHill_03-03-2026" / "Readme_20260303_RoundHillSpatial_.docx"
print(docx_text(f)[:600])

Captured output (excerpt, French original):
```
2026 03 03 - Round Hill Spatial
Equipe: Jean-Benoit Madore, Alexandre Langlois, Nicolas Marchand, Benjamin Imbach, Nicolas Allet,
Violaine Paquette, Megan Cramb
...Le drone a eu des problemes de connexion RTK et...nous avons decide d'annuler les vols planifies.
...le reste de l'equipe a effectue un transect de 18 points avec Dual KU, Radar K, SMP, SS et HS.
```
Confirms Day1's 18-point spatial transect count matches `Spatial_Round_Hill.xlsx` (18 data rows).